# Detecção de Fraude em Transações Bancárias

## Introdução

Esse projeto desenvolve um modelo de machine learning para detecção de fraudes em transações de cartão de crédito, utilizando uma base de dados altamente desbalanceada. A análise envolve exploração dos dados, construção e comparação de modelos, com foco em obter um bom equilíbrio entre identificação de fraudes e redução de erros.

# Etapas do código

### Entendimento do problema e dos dados
- leitura da base;
- análise de colunas, tipos, estatísticas descritivas e distribuição da variável alvo.

### Preparação dos dados
- verificação de valores nulos;
- remoção de duplicatas;
- justificativa da ausência de variáveis categóricas;
- uso de padronização para modelos sensíveis à escala via pipeline.

### Análise exploratória
- histogramas, boxplots, heatmap, scatter plot;
- correlação com a variável alvo;
- assimetria e curtose;
- comparação entre fraude e não fraude.

### Seleção e comparação de modelos
- Regressão Logística, Random Forest e XGBoost;
- divisão estratificada entre treino, validação e teste;
- validação cruzada;
- ajuste de hiperparâmetros.

### Treinamento e avaliação
- uso de métricas adequadas para base desbalanceada;
- comparação entre modelos;
- escolha do melhor modelo em validação.

### Validação final
- ajuste de threshold;
- re-treino em treino + validação;
- teste final em dados não vistos.

## Resumo dos resultados finais

Após a remoção de duplicatas, a base ficou com 283.726 registros, sendo:
- 283.253 transações normais
- 473 fraudes

Na modelagem:
- XGBoost foi o melhor algoritmo;
- F1 CV inicial = 0.7959
- melhor F1 após tuning = 0.7999

No conjunto de teste:
- com threshold 0.50, o modelo obteve precision = 0.9259, recall = 0.7895, F1 = 0.8523 e ROC AUC = 0.9799
- com threshold ajustado = 0.9535, a precision subiu para 0.9730, enquanto o recall caiu para 0.7579

Na prática, o modelo final generalizou bem e mostrou desempenho forte para um problema de fraude extremamente desbalanceado.


In [ ]:
# =========================================================
# BIBLIOTECAS
# =========================================================
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    RandomizedSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    average_precision_score,
    classification_report,
    ConfusionMatrixDisplay,
    roc_curve,
    precision_recall_curve
)

xgb_disponivel = True
try:
    from xgboost import XGBClassifier
except ImportError:
    xgb_disponivel = False
    print("\n[AVISO] XGBoost não encontrado no ambiente.")
    print("O fluxo seguirá com Regressão Logística e Random Forest.")

sns.set_theme(style="whitegrid")


## Entendimento do problema

O problema é de classificação binária com foco em detecção de fraude em transações de cartão de crédito.

A base é desbalanceada, então a análise não pode depender apenas de acurácia. 
Outras métricas como precision, recall, F1-score, ROC AUC e PR AUC são mais adequadas para medir o desempenho do modelo.


In [ ]:
# =========================================================
# CONFIGURAÇÕES GERAIS
# =========================================================
RODAR_EDA = True
RANDOM_STATE = 42
CV_SPLITS = 3
TAMANHO_AMOSTRA_TUNING = 60000


In [ ]:
# =========================================================
# DATASET
# =========================================================
df = pd.read_csv("Base_M43_Pratique_CREDIT_CARD_FRAUD.csv")

print("=" * 70)
print("DATASET CARREGADO")
print("=" * 70)
print(f"Linhas: {df.shape[0]:,}")
print(f"Colunas: {df.shape[1]}")

print("\nColunas:")
print(df.columns.tolist())

print("\n" + "=" * 70)
print("PRIMEIRAS LINHAS")
print("=" * 70)
print(df.head(10))


## Leitura inicial da base

A base original possui 284.807 registros e 31 colunas.

As variáveis presentes são:
- Time
- V1 até V28
- Amount
- Class

A variável Class representa o alvo do problema:
- 0 = transação normal
- 1 = fraude


In [ ]:
# =========================================================
# INFORMAÇÕES GERAIS 
# =========================================================

# Informações Gerais
print("\n" + "=" * 70)
print("INFORMAÇÕES GERAIS")
print("=" * 70)
print(df.info())

# Tipos de Dados
print("\n" + "=" * 70)
print("TIPOS DE DADOS")
print("=" * 70)
print(df.dtypes.value_counts())

# Valores Nulos
print("\n" + "=" * 70)
print("VALORES NULOS")
print("=" * 70)
print(df.isnull().sum())
print(f"\nTotal de valores nulos na base: {df.isnull().sum().sum()}")

# Duplicatas
duplicadas_antes = df.duplicated().sum()

print("\n" + "=" * 70)
print("DUPLICATAS")
print("=" * 70)
print(f"Quantidade de linhas duplicadas antes da remoção: {duplicadas_antes}")

df = df.drop_duplicates().copy()

print(f"Quantidade de linhas duplicadas após a remoção: {df.duplicated().sum()}")
print(f"Novo total de linhas após remoção: {df.shape[0]:,}")

# Estatísticas Descritivas
print("\n" + "=" * 70)
print("ESTATÍSTICAS DESCRITIVAS")
print("=" * 70)
print(df.describe().T)

# Distribuição da Variável alvo
print("\n" + "=" * 70)
print("DISTRIBUIÇÃO DA VARIÁVEL ALVO")
print("=" * 70)
print("Contagem:")
print(df["Class"].value_counts())

print("\nPercentual:")
print(df["Class"].value_counts(normalize=True) * 100)

### Preparação dos dados

### Valores faltantes

A base não apresentou valores nulos, então não foi necessário aplicar imputação.

Variáveis categóricas

Todas as variáveis são numéricas, portanto não foi necessário realizar codificação.

### Duplicatas

Foram identificadas 1.081 duplicatas, que foram removidas.
A remoção foi escolhida para evitar que o modelo aprenda padrões repetidos de forma indevida, o que poderia distorcer os resultados principalmente em uma base já desbalanceada.

Após isso, a base ficou com 283.726 registros.

### Variável alvo

A distribuição final ficou:

283.253 transações normais
473 fraudes

### Em percentual:

99,8333% classe 0
0,1667% classe 1

Esse desbalanceamento é o principal desafio do projeto.

In [ ]:
# =========================================================
# ANÁLISE EXPLORATÓRIA 
# =========================================================

if RODAR_EDA:
    
    # Distribuição da Classe Alvo
    plt.figure(figsize=(6, 4))
    sns.countplot(data=df, x="Class")
    plt.title("Distribuição da Classe Alvo")
    plt.show()

    # Distribuição da Variável Amount
    plt.figure(figsize=(8, 4))
    plt.hist(df["Amount"], bins=50)
    plt.title("Distribuição da Variável Amount")
    plt.show()

    # Boxplot da Variável Amount
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=df["Amount"])
    plt.title("Boxplot da Variável Amount")
    plt.show()

    # Amount por Classe
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df, x="Class", y="Amount")
    plt.title("Amount por Classe")
    plt.show()

    # Distribuição da Variável Time
    plt.figure(figsize=(8, 4))
    plt.hist(df["Time"], bins=50)
    plt.title("Distribuição da Variável Time")
    plt.show()

    corr_class = df.corr(numeric_only=True)["Class"].sort_values(ascending=False)

    print("\n" + "=" * 70)
    print("CORRELAÇÃO DAS VARIÁVEIS COM A CLASSE")
    print("=" * 70)
    print(corr_class)

    # Mapa de Correlação
    plt.figure(figsize=(12, 8))
    sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm", center=0)
    plt.title("Mapa de Correlação")
    plt.show()

    print("\n" + "=" * 70)
    print("TOP 10 CORRELAÇÕES POSITIVAS COM A CLASSE")
    print("=" * 70)
    print(corr_class.head(10))

    print("\n" + "=" * 70)
    print("TOP 10 CORRELAÇÕES NEGATIVAS COM A CLASSE")
    print("=" * 70)
    print(corr_class.tail(10))

    print("\n" + "=" * 70)
    print("COMPARAÇÃO ENTRE AS CLASSES")
    print("=" * 70)
    print(df.groupby("Class")[["Time", "Amount"]].agg(["mean", "median", "std", "min", "max"]))

    # Time por Classe
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df, x="Class", y="Time")
    plt.title("Time por Classe")
    plt.show()

    plt.figure(figsize=(8, 5))
    sns.scatterplot(
        data=df.sample(min(10000, len(df)), random_state=RANDOM_STATE),
        x="Time",
        y="Amount",
        hue="Class",
        alpha=0.5
    )
    plt.title("Relação entre Time e Amount")
    plt.show()

    print("\n" + "=" * 70)
    print("MEDIDAS DE ASSIMETRIA")
    print("=" * 70)
    print(df[["Time", "Amount"]].skew())

    print("\n" + "=" * 70)
    print("MEDIDAS DE CURTOSE")
    print("=" * 70)
    print(df[["Time", "Amount"]].kurtosis())


## Análise exploratória 

### Amount
A variável Amount apresentou:
- assimetria = 16.9788
- curtose = 844.4713

Isso indica uma distribuição extremamente assimétrica, com forte concentração em valores baixos e muitos outliers.

### Time
A variável `Time` apresentou:
- assimetria = -0.0356
- curtose = -1.2934

Ou seja, o comportamento temporal é muito menos extremo do que Amount.

### Correlações com a classe alvo
As maiores correlações positivas com Class foram:
- V11 = 0.1491
- V4 = 0.1293
- V2 = 0.0846

As maiores correlações negativas foram:
- V17 = -0.3135
- V14 = -0.2934
- V12 = -0.2507

Isso mostra que existem sinais relevantes na base, mas que o problema não pode ser resolvido por uma variável isolada. A decisão depende da combinação entre múltiplas features.


In [ ]:
# =========================================================
# SEPARAÇÃO ENTRE FEATURES E TARGET
# =========================================================
X = df.drop("Class", axis=1)
y = df["Class"]

print("\n" + "=" * 70)
print("SEPARAÇÃO FINAL ENTRE VARIÁVEIS")
print("=" * 70)
print(f"Shape de X: {X.shape}")
print(f"Shape de y: {y.shape}")


In [ ]:
# =========================================================
# DIVISÃO EM TREINO, VALIDAÇÃO E TESTE
# =========================================================
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

print("\n" + "=" * 70)
print("DIVISÃO DOS DADOS")
print("=" * 70)
print(f"Treino     : {X_train.shape[0]:,} linhas")
print(f"Validação  : {X_val.shape[0]:,} linhas")
print(f"Teste      : {X_test.shape[0]:,} linhas")


## Divisão dos dados

Os dados foram divididos em:
- treino: 170.235 linhas
- validação: 56.745 linhas
- teste: 56.746 linhas

A estratificação foi mantida para preservar a proporção da classe minoritária em todos os conjuntos.


In [ ]:
# =========================================================
# AMOSTRA PARA TUNING
# =========================================================

X_tune, _, y_tune, _ = train_test_split(
    X_train,
    y_train,
    train_size=min(TAMANHO_AMOSTRA_TUNING, len(X_train)),
    stratify=y_train,
    random_state=RANDOM_STATE
)

print("\n" + "=" * 70)
print("AMOSTRA PARA TUNING")
print("=" * 70)
print(f"X_tune: {X_tune.shape}")
print(f"y_tune: {y_tune.shape}")


## Amostra para tuning

Para equilibrar custo computacional e qualidade analítica, foi usada uma amostra de 60.000 registros para a fase de tuning. Isso deixou o processo mais leve sem comprometer a estrutura metodológica.


In [ ]:
# =========================================================
# RAZÃO DE DESBALANCEAMENTO
# =========================================================
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
scale_pos_weight = n_neg / n_pos

print("\n" + "=" * 70)
print("RAZÃO DE DESBALANCEAMENTO")
print("=" * 70)
print(f"Negativos no treino: {n_neg:,}")
print(f"Positivos no treino: {n_pos:,}")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")


## Peso da classe minoritária

No conjunto de treino foram observados:
- 169.951 negativos
- 284 positivos

Com isso, o scale_pos_weight calculado foi 598,42, valor usado no XGBoost para reforçar o aprendizado da classe fraude.


In [ ]:
# =========================================================
# MODELOS INICIAIS
# =========================================================
pipeline_logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1200,
        solver="liblinear",
        random_state=RANDOM_STATE
    ))
])

rf = RandomForestClassifier(
    n_estimators=150,
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    max_depth=12
)

if xgb_disponivel:
    xgb = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_estimators=180,
        learning_rate=0.08,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        n_jobs=-1
    )

modelos = {
    "Regressão Logística": pipeline_logreg,
    "Random Forest": rf
}

if xgb_disponivel:
    modelos["XGBoost"] = xgb

print("\n" + "=" * 70)
print("MODELOS SELECIONADOS")
print("=" * 70)
for nome in modelos:
    print("-", nome)


## Seleção de modelos

Foram testados três algoritmos:
- Regressão Logística
- Random Forest
- XGBoost

A escolha foi dos modelos foi baseado em que:
- a Regressão Logística funciona como baseline interpretável;
- o Random Forest lida bem com não linearidades;
- o XGBoost tende a performar muito bem em bases tabulares desbalanceadas.


In [ ]:
# =========================================================
# FUNÇÃO DE AVALIAÇÃO
# =========================================================
def avaliar_modelo(nome_modelo, modelo, X_treino, y_treino, X_avaliacao, y_avaliacao):
    modelo.fit(X_treino, y_treino)

    y_pred = modelo.predict(X_avaliacao)
    y_prob = modelo.predict_proba(X_avaliacao)[:, 1]

    resultados = {
        "Modelo": nome_modelo,
        "Accuracy": accuracy_score(y_avaliacao, y_pred),
        "Precision": precision_score(y_avaliacao, y_pred, zero_division=0),
        "Recall": recall_score(y_avaliacao, y_pred, zero_division=0),
        "F1": f1_score(y_avaliacao, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_avaliacao, y_prob),
        "PR_AUC": average_precision_score(y_avaliacao, y_prob),
        "Log_Loss": log_loss(y_avaliacao, y_prob)
    }

    return resultados, y_pred, y_prob


In [ ]:
# =========================================================
# CROSS VALIDATION
# =========================================================
cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

metricas_cv = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

resultados_cv = []

print("\n" + "=" * 70)
print("CROSS VALIDATION - MODELOS INICIAIS")
print("=" * 70)

for nome, modelo in modelos.items():
    scores = cross_validate(
        modelo,
        X_tune,
        y_tune,
        cv=cv,
        scoring=metricas_cv,
        n_jobs=-1
    )

    resultados_cv.append({
        "Modelo": nome,
        "Accuracy CV": scores["test_accuracy"].mean(),
        "Precision CV": scores["test_precision"].mean(),
        "Recall CV": scores["test_recall"].mean(),
        "F1 CV": scores["test_f1"].mean(),
        "ROC_AUC CV": scores["test_roc_auc"].mean()
    })

df_cv = pd.DataFrame(resultados_cv).sort_values(by="F1 CV", ascending=False)
print(df_cv)


## 9. Validação cruzada inicial

Na comparação inicial dos modelos, os resultados médios foram:

- **XGBoost:** `F1 CV = 0.7959` | `ROC AUC CV = 0.9676`
- **Random Forest:** `F1 CV = 0.7833` | `ROC AUC CV = 0.9308`
- **Regressão Logística:** `F1 CV = 0.0753` | `ROC AUC CV = 0.9479`

### Insight
A Regressão Logística ficou muito atrás no F1-score, mostrando baixa efetividade para a classe fraudulenta. Random Forest e XGBoost foram bem superiores, com vantagem inicial para o XGBoost.


In [ ]:
# =========================================================
# AJUSTE DE HIPERPARÂMETROS 
# =========================================================
print("\n" + "=" * 70)
print("AJUSTE DE HIPERPARÂMETROS")
print("=" * 70)

param_logreg = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__penalty": ["l1", "l2"]
}

logreg_search = RandomizedSearchCV(
    estimator=pipeline_logreg,
    param_distributions=param_logreg,
    n_iter=4,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)
logreg_search.fit(X_tune, y_tune)

print("\nMelhores parâmetros - Regressão Logística:")
print(logreg_search.best_params_)
print(f"Melhor F1 CV: {logreg_search.best_score_:.4f}")

param_rf = {
    "n_estimators": [120, 150, 200],
    "max_depth": [8, 12, 16],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"]
}

rf_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_rf,
    n_iter=6,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)
rf_search.fit(X_tune, y_tune)

print("\nMelhores parâmetros - Random Forest:")
print(rf_search.best_params_)
print(f"Melhor F1 CV: {rf_search.best_score_:.4f}")

if xgb_disponivel:
    param_xgb = {
        "n_estimators": [120, 180, 220],
        "max_depth": [3, 4, 5],
        "learning_rate": [0.05, 0.08, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "gamma": [0, 1],
        "min_child_weight": [1, 3]
    }

    xgb_search = RandomizedSearchCV(
        estimator=xgb,
        param_distributions=param_xgb,
        n_iter=6,
        scoring="f1",
        cv=cv,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=1
    )
    xgb_search.fit(X_tune, y_tune)

    print("\nMelhores parâmetros - XGBoost:")
    print(xgb_search.best_params_)
    print(f"Melhor F1 CV: {xgb_search.best_score_:.4f}")


## Ajuste de hiperparâmetros

Depois do tuning, os melhores resultados foram:

- Regressão Logística: melhor F1 CV = 0.0870
- Random Forest: melhor F1 CV = 0.7877
- XGBoost: melhor F1 CV = 0.7999

### Melhor configuração do XGBoost
- subsample = 0.8
- n_estimators = 220
- min_child_weight = 1
- max_depth = 5
- learning_rate = 0.1
- gamma = 1
- colsample_bytree = 1.0

O tuning confirmou que o XGBoost continuou sendo o modelo mais forte.


In [ ]:
# =========================================================
# VALIDAÇÃO DOS MODELOS
# =========================================================
modelos_ajustados = {
    "Regressão Logística": logreg_search.best_estimator_,
    "Random Forest": rf_search.best_estimator_
}

if xgb_disponivel:
    modelos_ajustados["XGBoost"] = xgb_search.best_estimator_

resultados_validacao = []
preds_validacao = {}
probs_validacao = {}

print("\n" + "=" * 70)
print("AVALIAÇÃO DOS MODELOS AJUSTADOS - VALIDAÇÃO")
print("=" * 70)

for nome, modelo in modelos_ajustados.items():
    res, y_pred_val, y_prob_val = avaliar_modelo(
        nome, modelo, X_train, y_train, X_val, y_val
    )
    resultados_validacao.append(res)
    preds_validacao[nome] = y_pred_val
    probs_validacao[nome] = y_prob_val

df_validacao = pd.DataFrame(resultados_validacao).sort_values(
    by=["F1", "ROC_AUC", "Recall"],
    ascending=False
)
print(df_validacao)


## Avaliação dos modelos na validação

### XGBoost
- Accuracy = 0.999648
- Precision = 0.920455
- ROC AUC = 0.972601
- PR AUC = 0.882889
- Log Loss = 0.002446

### Random Forest
- Accuracy = 0.999366
- Precision = 0.784314
- ROC AUC = 0.975608
- PR AUC = 0.814351
- Log Loss = 0.021147

### Regressão Logística
- Accuracy = 0.973936
- Precision = 0.054662
- ROC AUC = 0.979177
- PR AUC = 0.795212
- Log Loss = 0.104468

### Insight
O XGBoost foi o melhor modelo na validação. Ele combinou alta precisão, PR AUC mais forte e log loss muito baixo, além de manter ótimo desempenho geral.


In [ ]:
# =========================================================
# ESCOLHA DO MELHOR MODELO
# =========================================================
melhor_nome = df_validacao.iloc[0]["Modelo"]
melhor_modelo = modelos_ajustados[melhor_nome]

print("\n" + "=" * 70)
print("MELHOR MODELO EM VALIDAÇÃO")
print("=" * 70)
print(f"Melhor modelo: {melhor_nome}")


## Melhor modelo

Com base no conjunto de validação, o XGBoost foi escolhido como modelo final.


In [ ]:
# =========================================================
# OTIMIZAÇÃO DE LIMIAR
# =========================================================
y_prob_val_best = probs_validacao[melhor_nome]

precision_curve, recall_curve, thresholds = precision_recall_curve(y_val, y_prob_val_best)
f1_scores_curve = 2 * (precision_curve[:-1] * recall_curve[:-1]) / (
    precision_curve[:-1] + recall_curve[:-1] + 1e-10
)

best_idx = np.argmax(f1_scores_curve)
best_threshold = thresholds[best_idx]

print("\n" + "=" * 70)
print("OTIMIZAÇÃO DE LIMIAR")
print("=" * 70)
print(f"Melhor threshold: {best_threshold:.4f}")
print(f"F1: {f1_scores_curve[best_idx]:.4f}")
print(f"Precision: {precision_curve[:-1][best_idx]:.4f}")
print(f"Recall: {recall_curve[:-1][best_idx]:.4f}")


## Ajuste de threshold

O melhor threshold encontrado na validação foi 0.9535.

Nesse ponto, o modelo alcançou:
- F1 = 0.8966
- Precision = 0.9750
- Recall = 0.8298

Isso mostra que, ao elevar o limiar, o modelo fica mais conservador: ele aumenta a confiança nas previsões positivas, mas pode deixar escapar mais fraudes.

In [ ]:
# =========================================================
# TREINO FINAL
# =========================================================
X_dev = pd.concat([X_train, X_val], axis=0)
y_dev = pd.concat([y_train, y_val], axis=0)

modelo_final = clone(melhor_modelo)
modelo_final.fit(X_dev, y_dev)

print("\n" + "=" * 70)
print("MODELO FINAL TREINADO")
print("=" * 70)
print(f"Modelo final: {melhor_nome}")


In [ ]:
# =========================================================
# TESTE FINAL
# =========================================================
y_pred_test_default = modelo_final.predict(X_test)
y_prob_test = modelo_final.predict_proba(X_test)[:, 1]
y_pred_test_ajustado = (y_prob_test >= best_threshold).astype(int)

res_default = {
    "Threshold": 0.50,
    "Accuracy": accuracy_score(y_test, y_pred_test_default),
    "Precision": precision_score(y_test, y_pred_test_default, zero_division=0),
    "Recall": recall_score(y_test, y_pred_test_default, zero_division=0),
    "F1": f1_score(y_test, y_pred_test_default, zero_division=0),
    "ROC_AUC": roc_auc_score(y_test, y_prob_test),
    "PR_AUC": average_precision_score(y_test, y_prob_test),
    "Log_Loss": log_loss(y_test, y_prob_test)
}

res_ajustado = {
    "Threshold": best_threshold,
    "Accuracy": accuracy_score(y_test, y_pred_test_ajustado),
    "Precision": precision_score(y_test, y_pred_test_ajustado, zero_division=0),
    "Recall": recall_score(y_test, y_pred_test_ajustado, zero_division=0),
    "F1": f1_score(y_test, y_pred_test_ajustado, zero_division=0),
    "ROC_AUC": roc_auc_score(y_test, y_prob_test),
    "PR_AUC": average_precision_score(y_test, y_prob_test),
    "Log_Loss": log_loss(y_test, y_prob_test)
}

df_teste_final = pd.DataFrame([res_default, res_ajustado])

print("\n" + "=" * 70)
print("RESULTADOS FINAIS - CONJUNTO DE TESTE")
print("=" * 70)
print(df_teste_final)


## Teste final em dados não vistos

### Threshold padrão = 0.50
- Accuracy = 0.999542
- Precision = 0.925926
- Recall = 0.7895
- F1 = 0.8523
- ROC AUC = 0.979858
- PR AUC = 0.824581
- Log Loss = 0.003043

### Threshold ajustado = 0.9535
- Accuracy = 0.999559
- Precision = 0.972973
- Recall = 0.7579
- F1 = 0.8521
- ROC AUC = 0.979858
- PR AUC = 0.824581
- Log Loss = 0.003043

### Leitura final
Com threshold 0.50, o modelo ficou mais equilibrado.  
Com threshold ajustado, a precisão subiu, mas o recall caiu.

Como o objetivo do caso é minimizar tanto falsos positivos quanto falsos negativos, o cenário com threshold 0.50 ficou mais consistente.


In [ ]:
# =========================================================
# MATRIZES DE CONFUSÃO
# =========================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_test_default, ax=axes[0], colorbar=False
)
axes[0].set_title("Threshold 0.50")

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_test_ajustado, ax=axes[1], colorbar=False
)
axes[1].set_title(f"Threshold {best_threshold:.2f}")

plt.tight_layout()
plt.show()


## Matrizes de confusão

### Threshold 0.50
TN = 56.645
FP = 6
FN = 20
TP = 75

Aqui o modelo está mais equilibrado: consegue pegar mais fraudes (maior TP) e ainda mantém poucos falsos positivos. Ou seja, detecta bem sem atrapalhar muito o cliente.

### Threshold 0.95
- TN = 56.649
- FP = 2
- FN = 23
- TP = 72

Com o threshold mais alto, o modelo fica mais conservador: erra menos ao acusar fraude (menos FP), mas deixa passar mais fraudes (mais FN).

Fica claro o trade-off:
0.50 → mais fraudes detectadas (melhor recall)
0.95 → menos erro ao acusar fraude (melhor precision)

No geral, o threshold 0.50 fica mais equilibrado para o problema.

In [ ]:
# =========================================================
# RELATÓRIO DE CLASSIFICAÇÃO
# =========================================================
print("\n" + "=" * 70)
print("RELATÓRIO DE CLASSIFICAÇÃO - THRESHOLD PADRÃO")
print("=" * 70)
print(classification_report(y_test, y_pred_test_default, digits=4))

print("\n" + "=" * 70)
print("RELATÓRIO DE CLASSIFICAÇÃO - THRESHOLD AJUSTADO")
print("=" * 70)
print(classification_report(y_test, y_pred_test_ajustado, digits=4))


In [ ]:
# =========================================================
# CURVAS
# =========================================================
fpr, tpr, _ = roc_curve(y_test, y_prob_test)
roc_auc = roc_auc_score(y_test, y_prob_test)

plt.figure(figsize=(8, 5))
plt.plot(fpr, tpr, label=f"{melhor_nome} (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("Curva ROC")
plt.legend()
plt.show()

precision_test, recall_test, _ = precision_recall_curve(y_test, y_prob_test)
pr_auc = average_precision_score(y_test, y_prob_test)

plt.figure(figsize=(8, 5))
plt.plot(recall_test, precision_test, label=f"{melhor_nome} (PR AUC = {pr_auc:.4f})")
plt.title("Curva Precision-Recall")
plt.legend()
plt.show()


## Curvas ROC e Precision-Recall

O modelo final apresentou:
- ROC AUC = 0.9799
- PR AUC = 0.8246

A curva ROC mostra excelente capacidade de separação entre classes.  
A curva Precision-Recall confirma que o modelo se comporta muito bem na classe minoritária, o que é essencial neste problema.


In [ ]:
# =========================================================
# IMPORTÂNCIA DAS FEATURES
# =========================================================
if melhor_nome == "Random Forest":
    importancias = pd.DataFrame({
        "Feature": X.columns,
        "Importance": modelo_final.feature_importances_
    }).sort_values("Importance", ascending=False)

    print(importancias.head(15))

    plt.figure(figsize=(8, 6))
    sns.barplot(data=importancias.head(15), x="Importance", y="Feature")
    plt.title("Top 15 Features - Random Forest")
    plt.show()

elif melhor_nome == "XGBoost":
    importancias = pd.DataFrame({
        "Feature": X.columns,
        "Importance": modelo_final.feature_importances_
    }).sort_values("Importance", ascending=False)

    print(importancias.head(15))

    plt.figure(figsize=(8, 6))
    sns.barplot(data=importancias.head(15), x="Importance", y="Feature")
    plt.title("Top 15 Features - XGBoost")
    plt.show()

elif melhor_nome == "Regressão Logística":
    coeficientes = pd.DataFrame({
        "Feature": X.columns,
        "Coef_Abs": np.abs(modelo_final.named_steps["model"].coef_[0])
    }).sort_values("Coef_Abs", ascending=False)

    print(coeficientes.head(15))

    plt.figure(figsize=(8, 6))
    sns.barplot(data=coeficientes.head(15), x="Coef_Abs", y="Feature")
    plt.title("Top 15 Features - Regressão Logística")
    plt.show()


## Importância das variáveis

As features mais relevantes no XGBoost foram:

1. V14 = 0.5064
2. V4 = 0.0675
3. V8 = 0.0457
4. V10 = 0.0388
5. V12 = 0.0349

Também se destacaram:
V19, V13, V3, V15, V11, V17, V7, V20, V22 e Amount.

A variável V14 apareceu com peso muito dominante, sugerindo forte relevância nessa dimensão transformada da base.


## Conclusão final

O projeto cumpriu os requisitos centrais da tarefa e chegou a uma solução robusta para o caso proposto.

### Principais conclusões
- a base foi preparada corretamente;
- a EDA confirmou forte desbalanceamento e alta complexidade do problema;
- os modelos foram comparados com métricas adequadas;
- o XGBoost foi o melhor algoritmo;
- no teste, o modelo atingiu F1 = 0.8523, precision = 0.9259, recall = 0.7895, ROC AUC = 0.9799 e PR AUC = 0.8246;
- o ajuste de threshold mostrou que é possível tornar o modelo mais conservador, mas com perda de recall.

### Decisão recomendada
Para este case, a configuração com threshold 0.50 ficou mais equilibrada para o objetivo de negócio, pois mantém boa capacidade de detecção de fraude com um número muito baixo de falsos positivos.
